# Is the tracker better than the static motif pipelines?

Mouse 720762, deep prelimbic LFP (PL5). Two questions:

1. **States.** Does the tracker's output (motif expression, baseline, spread, presence) follow isoflurane level,
   pupil and running better than the static motifs (original strict and relaxed), and better than plain band power?
   Do the motif poles themselves move with state?
2. **Neurons.** On like-for-like tests, does the tracker's frame (sliding baseline) track opponent neuron groups
   better, after controlling for broadband power? Does a motif's baseline add anything?

Rules used throughout:
- Part 1 (states) uses every second, stimulation blocks included: spontaneous seconds cover only ~15 min of the
  awake period, and the within-maintenance dose changes (1.5 -> 3.5 %) happen inside the isoflurane electrical
  block. Part 2 (neurons) uses spontaneous seconds only, as before.
- Slow targets (dose, pupil, running) are tested one feature at a time, not with multi-feature regression: with
  one session there are only a handful of independent slow stretches, so a regression on many slow features fits
  anything (see 1.1). Nulls circularly shift the target by at least 5 min.
- Fast targets (spiking) use held-out ridge regression with interleaved 60-s blocks (5 folds, 10 s trimmed at
  block edges), and only per-second predictors, for the same reason.
- Static motif directions come from the 1 Nov run; their per-second expression is the projection of the
  band envelopes (z-scored over the session) onto the direction. Using one frame for the whole session gives
  the static motifs their best chance to carry between-state information; in their native form they are
  z-scored within a state and carry none.

In [2]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import spearmanr
REPO = Path("/Users/namanrudrakshi/Developer/dc/dc-allen-data-acquire")
sys.path.insert(0, str(REPO / "code" / "python"))
from src import motif_tracking as mt, spike_validation as sv, session_data as sd
import warnings; warnings.filterwarnings("ignore")
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False})

NWB = REPO / "data" / "raw" / "720762-20240221-v001.nwb"
OUT = REPO / "data" / "out" / "2026-11-10_tracker_vs_static"; OUT.mkdir(parents=True, exist_ok=True)
FIG = OUT / "figures"; FIG.mkdir(exist_ok=True)
def save(fig, name): fig.savefig(FIG / f"{name}.png", dpi=200, bbox_inches="tight", facecolor="white")

res = mt.load(REPO / "data" / "processed" / "motif_tracking" / "720762" / "lfp_PL5.pkl")
sig = mt.prepare(NWB, "lfp_PL5", ("lfp", "probeF", "PL5"), res.config)
info, (run_t, run), (pupil_t, pupil) = mt.behaviour(NWB)
t = sig.t
K = len(res.library)
U = np.array([m.template for m in res.library])
E = res.second[[f"expr_M{i + 1}" for i in range(K)]].to_numpy()
Bm = res.second[[f"base_M{i + 1}" for i in range(K)]].to_numpy()
Sp = res.second[[f"spread_M{i + 1}" for i in range(K)]].to_numpy()

R = json.load(open(REPO / "data" / "out" / "2026-11-01_mouse720762" / "results.json"))
runs = {(r["window"], r["version"]): np.array(r["motifs"], float)[:, :r["n_motifs"]] for r in R["motif_runs"] if r["signal"] == "lfp_PL5"}
unit = lambda v: v / np.linalg.norm(v)
STATIC = {"Original (awake)": unit(runs["awake", "strict"][:, 0]),
          "Relaxed (awake)": unit(runs["awake", "relaxed"][:, 0]),
          "Relaxed (whole session)": unit(runs["full", "relaxed"][:, 0]),
          "Relaxed (isoflurane)": unit(runs["isoflurane", "relaxed"][:, 0])}
BROAD = unit(np.ones(sig.Z.shape[1]))
# match static directions to tracker motifs (|cos|)
match = pd.DataFrame({name: np.abs(U @ u) for name, u in STATIC.items()}, index=[f"M{i + 1}" for i in range(K)])
match.loc["broadband"] = [abs(BROAD @ u) for u in STATIC.values()]

# per-second behaviour and dose
csum = np.r_[0, np.cumsum(run)]
lo, hi = np.searchsorted(run_t, t - 0.5), np.searchsorted(run_t, t + 0.5)
speed = (csum[hi] - csum[lo]) / np.maximum(hi - lo, 1)
okp = np.isfinite(pupil)
pup = np.interp(t, pupil_t[okp], pupil[okp])
pup = pd.Series(pup).rolling(5, center=True, min_periods=1).mean().to_numpy()
iso = np.interp(t, info.iso_t, info.iso_level)
state = sd.state_of_time(t, info)
spont = sv.spontaneous_mask(t, info)
ok_tr = np.isfinite(E).all(1) & np.isfinite(Bm).all(1) & np.isfinite(Sp).all(1)
print(f"{K} tracker motifs; seconds: {t.size}; spontaneous with tracker output: {(spont & ok_tr).sum()}")
match.round(2)

9 tracker motifs; seconds: 7088; spontaneous with tracker output: 2613


,Original (awake),Relaxed (awake),Relaxed (whole session),Relaxed (isoflurane)
M1,0.68,0.65,0.05,0.03
M2,0.27,0.28,0.68,0.93
M3,0.57,0.57,0.58,0.33
M4,0.34,0.31,0.38,0.46
M5,0.25,0.27,0.67,0.65
M6,0.43,0.42,0.12,0.35
M7,0.26,0.24,0.72,0.55
M8,0.75,0.76,0.04,0.00
M9,0.52,0.52,0.37,0.46
broadband,0.57,0.54,0.01,0.07


## 1.1 Which outputs follow state? One table for all methods

**Why not regression.** A first pass predicted each target from each method's features with held-out ridge
regression. With interleaved 60-s folds, the slow (5-min) tracker features fitted even time-shifted targets
(null R² up to 0.6); with contiguous folds every method went negative, because one session holds only a few
independent slow stretches. Neither is a fair test, so each method is scored by its single best feature.

**Test.** For each method, the largest |Spearman r| between any one of its per-second features and the target.
Null: the same maximum after circularly shifting the target by at least 5 min (200 shifts), so methods with
more features are not favoured.

Targets: isoflurane level over the whole session, isoflurane level within maintenance (1.5–3.5 %), pupil within
awake, pupil within maintenance, running speed within awake.

Methods:
- **Original**: the awake strict motif (1 feature);
- **Relaxed**: the three relaxed motifs (3);
- **Tracker expression** (9), **tracker baseline** (9), **tracker spread** (9);
- controls: **10 bands** (raw envelopes), **10 bands, 5-min baseline** (the tracker's sliding baseline before
  any motif), and **Relaxed + sliding baseline** (the relaxed directions with the tracker's baseline idea).

In [3]:
Zg = sig.Z
Srel = np.column_stack([STATIC[k] for k in STATIC if k.startswith("Relaxed")])
FEATS = {
    "Original": Zg @ STATIC["Original (awake)"][:, None],
    "Relaxed": Zg @ Srel,
    "Tracker expression": E,
    "Tracker baseline": Bm,
    "Tracker spread": Sp,
    "10 bands": Zg,
    "10 bands, 5-min baseline": sig.base,
    "Relaxed + sliding baseline": np.column_stack([sig.resid @ Srel, sig.base @ Srel]),
}
FCOL = {"Original": "#9AA5B1", "Relaxed": "#2F7F8F", "Tracker expression": "#E0A030", "Tracker baseline": "#C2504A",
        "Tracker spread": "#8E3B2E", "10 bands": "#D5DBE2", "10 bands, 5-min baseline": "#B7A4C9",
        "Relaxed + sliding baseline": "#7FB3A8"}
FNAMES = {"Original": ["Original"], "Relaxed": [k for k in STATIC if k.startswith("Relaxed")],
          "Tracker expression": [f"M{i + 1} expression" for i in range(K)], "Tracker baseline": [f"M{i + 1} baseline" for i in range(K)],
          "Tracker spread": [f"M{i + 1} spread" for i in range(K)], "10 bands": [f"band {i + 1}" for i in range(10)],
          "10 bands, 5-min baseline": [f"band {i + 1} baseline" for i in range(10)],
          "Relaxed + sliding baseline": [f"{k} residual" for k in STATIC if k.startswith("Relaxed")] + [f"{k} baseline" for k in STATIC if k.startswith("Relaxed")]}
all_ok = ok_tr & np.isfinite(sig.base).all(1) & np.isfinite(sig.resid).all(1)
TARGETS = {
    "Isoflurane level\n(whole session)": (iso, all_ok),
    "Isoflurane level\n(within maintenance)": (iso, all_ok & (state == "isoflurane")),
    "Pupil\n(within awake)": (pup, all_ok & (state == "awake")),
    "Pupil\n(within maintenance)": (pup, all_ok & (state == "isoflurane")),
    "Running speed\n(within awake)": (speed, all_ok & (state == "awake")),
}
def ranks(A):
    A = np.asarray(A, float)
    A = A if A.ndim == 2 else A[:, None]
    r = np.argsort(np.argsort(A, axis=0), axis=0).astype(float)
    return (r - r.mean(0)) / (r.std(0) + 1e-12)
def best_feature(F, y, rng, n_shift=200, min_shift=300):
    RF, ry = ranks(F), ranks(y)[:, 0]
    rs = RF.T @ ry / ry.size
    ks = rng.integers(min_shift, ry.size - min_shift, n_shift)
    null = np.array([np.abs(RF.T @ np.roll(ry, k) / ry.size).max() for k in ks])
    j = int(np.argmax(np.abs(rs)))
    return j, float(rs[j]), float(np.percentile(null, 95)), float((1 + (null >= abs(rs[j])).sum()) / (n_shift + 1))

rows = []
rng = np.random.default_rng(0)
for tname, (y, m) in TARGETS.items():
    for fname, F in FEATS.items():
        j, r, n95, p = best_feature(F[m], y[m], rng)
        rows.append({"target": tname, "method": fname, "n_features": F.shape[1], "seconds": int(m.sum()),
                     "best feature": FNAMES[fname][j], "r": r, "|r|": abs(r), "null95 of max |r|": n95, "p": p})
score = pd.DataFrame(rows)
score.assign(target=score.target.str.replace("\n", " ")).to_csv(OUT / "1_state_association.csv", index=False)
score.pivot_table(index="method", columns="target", values="|r|", sort=False).round(2)

target,Isoflurane level\n(whole session),Isoflurane level\n(within maintenance),Pupil\n(within awake),Pupil\n(within maintenance),Running speed\n(within awake)
method,,,,,
Original,0.30,0.41,0.06,0.38,0.20
Relaxed,0.69,0.41,0.19,0.37,0.21
Tracker expression,0.04,0.07,0.07,0.11,0.11
Tracker baseline,0.80,0.72,0.59,0.67,0.26
Tracker spread,0.75,0.36,0.39,0.38,0.21
10 bands,0.76,0.51,0.24,0.58,0.23
"10 bands, 5-min baseline",0.79,0.65,0.51,0.69,0.28
Relaxed + sliding baseline,0.72,0.64,0.15,0.49,0.27


In [4]:
fig, ax = plt.subplots(figsize=(14, 4.8))
tg, fs = list(TARGETS), list(FEATS)
w = 0.8 / len(fs)
for j, f in enumerate(fs):
    for i, tn in enumerate(tg):
        r = score[(score.target == tn) & (score.method == f)].iloc[0]
        x = i + (j - (len(fs) - 1) / 2) * w
        sig_ = r.p <= 0.05
        ax.bar(x, r["|r|"], w * 0.92, color=FCOL[f] if sig_ else "white", edgecolor=FCOL[f], lw=1.2,
               label=f"{f} ({r.n_features})" if i == 0 else None)
        ax.plot([x - w / 2, x + w / 2], [r["null95 of max |r|"]] * 2, color="#444444", lw=1)
ax.set_xticks(range(len(tg)), tg)
ax.set_ylabel("best single feature, |Spearman r|"); ax.set_ylim(0, 1.0)
ax.legend(fontsize=8, ncol=4, frameon=False, loc="upper right", title="method (number of features)", title_fontsize=8)
save(fig, "Q1a_state_association"); plt.show()

## 1.2 How fast does each output register isoflurane?

Every output is turned into a causal 30-s trailing mean (only past seconds) and expressed in units of the last
2 minutes before switch-on (z against those seconds' mean and per-second SD). Latency = first second after
switch-on at which the trailing mean is more than 3 SD from the pre-switch mean and stays there for 30 s.

The tracker's baseline is left out of the latency table: its segments are cut at the known switch-on time, so it
is told when isoflurane starts. The question here is which *direction* reacts first.

In [5]:
def trailing(v, n=30):
    c = np.r_[0, np.cumsum(np.nan_to_num(v))]
    k = np.arange(v.size)
    lo_ = np.maximum(k + 1 - n, 0)
    return (c[k + 1] - c[lo_]) / (k + 1 - lo_)

pre = (t >= info.iso_on - 120) & (t < info.iso_on)
TRACES = {"Isoflurane level": iso, "Pupil": pup, "Running speed": speed,
          "Broadband power": Zg @ BROAD,
          **{k: Zg @ u for k, u in STATIC.items()},
          **{f"Tracker M{i + 1} direction": Zg @ U[i] for i in range(3)}}
win = (t >= info.iso_on - 600) & (t <= info.iso_on + 900)
lat_rows, ztr = [], {}
for name, v in TRACES.items():
    tr = trailing(v)
    z = (tr - v[pre].mean()) / (v[pre].std() + 1e-12)
    out = np.abs(z) > 3
    after = np.flatnonzero(t >= info.iso_on)
    lat = np.nan
    for k in after:
        if k + 30 < t.size and out[k:k + 30].all():
            lat = t[k] - info.iso_on; break
    ztr[name] = z
    lat_rows.append({"output": name, "latency after switch-on (s)": lat,
                     "direction of change": np.sign(np.nanmean(z[(t > info.induction_end) & (t < info.induction_end + 300)]))})
lat = pd.DataFrame(lat_rows); lat.to_csv(OUT / "1_induction_latency.csv", index=False)
lat

,output,latency after switch-on (s),direction of change
0,Isoflurane level,0.677446,1.0
1,Pupil,59.677251,-1.0
2,Running speed,97.677113,-1.0
3,Broadband power,84.677157,-1.0
4,Original (awake),96.677115,-1.0
5,Relaxed (awake),97.677113,-1.0
6,Relaxed (whole session),102.677101,-1.0
7,Relaxed (isoflurane),61.677246,-1.0
8,Tracker M1 direction,87.677146,-1.0
9,Tracker M2 direction,52.677271,1.0


In [6]:
groups = [("Dose and behaviour", ["Isoflurane level", "Pupil", "Running speed"], ["#555555", "#8E5EA2", "#E0A030"]),
          ("Static motif directions", ["Broadband power", "Original (awake)", "Relaxed (awake)", "Relaxed (whole session)", "Relaxed (isoflurane)"],
           ["#BBBBBB", "#9AA5B1", "#2F7F8F", "#4C8DC9", "#C2504A"]),
          ("Tracker directions and baseline", ["Tracker M1 direction", "Tracker M2 direction", "Tracker M3 direction"], ["#1D232B", "#E0A030", "#2F7F8F"])]
fig, axes = plt.subplots(3, 1, figsize=(13, 8.5), sharex=True)
tt = t[win] - info.iso_on
for ax, (title, names, cols) in zip(axes, groups):
    for n, c in zip(names, cols):
        L = lat.set_index("output").loc[n, "latency after switch-on (s)"]
        ax.plot(tt, np.clip(ztr[n][win], -15, 15), color=c, lw=1.3, label=f"{n} (changes at {L:.0f} s)" if np.isfinite(L) else f"{n} (no sustained change)")
    if title.startswith("Tracker"):
        for i, c in zip(range(3), cols):
            b = Bm[:, i].copy(); x_ = Zg @ U[i]
            ax.plot(tt, np.clip((b - x_[pre].mean()) / x_[pre].std(), -15, 15)[win], color=c, lw=2.2, ls="--", alpha=0.7,
                    label=f"Tracker M{i + 1} baseline (centred 5 min, cut at switch-on)")
    ax.axvspan(0, info.induction_end - info.iso_on, color="0.9", lw=0, zorder=0)
    ax.axvline(0, color="k", lw=0.8)
    ax.axhspan(-3, 3, color="#F3F3F3", lw=0, zorder=0)
    ax.set_title(title, loc="left", fontsize=10); ax.set_ylabel("z (pre-switch units)")
    ax.legend(fontsize=7.5, frameon=False, ncol=2, loc="lower left")
    ax.set_ylim(-16, 16)
axes[-1].set_xlabel("seconds from isoflurane switch-on (grey column: induction; grey band: ±3 SD of the 2 min before switch-on; clipped at ±15)")
fig.tight_layout(); save(fig, "Q1b_induction_onset"); plt.show()

## 1.3 Do the poles move? Expression along each motif axis, window by window

Same 5-min windows as the tracker (60-s steps), all seconds, in the session-wide frame (no sliding baseline, so
the raw movement is visible). In each window, along the motif axis:
- **+ pole** = mean of the seconds above the window median; **− pole** = mean of the seconds below it;
- **midpoint** = (+ pole + − pole) / 2 (the baseline);
- **separation** = + pole − − pole (how far apart the poles are);
- **asymmetry** = (+ pole − median) − (median − − pole): positive when the + side has the longer tail.

This splits movement into three kinds: both poles slide together (midpoint), they move apart or together
(separation), or one moves on its own (asymmetry). Each measure is then correlated with dose and behaviour at
the window level, with a null that circularly shifts the window series (by at least 10 windows).

In [7]:
AXES = {"Tracker M1 (broadband)": U[0], "Tracker M2": U[1], "Tracker M3": U[2], "Original (awake)": STATIC["Original (awake)"]}
starts = mt.window_starts(sig, res.config)
centres = starts + res.config.window / 2
wst = sd.state_of_time(centres, info)
def win_measures(x):
    rows = []
    for a in starts:
        sel = (t >= a) & (t < a + res.config.window) & np.isfinite(x)
        if sel.sum() < 150:
            rows.append({k: np.nan for k in ("plus", "minus", "mid", "sep", "asym", "iso", "pupil", "speed")}); continue
        v = x[sel]; med = np.median(v)
        p, n_ = v[v > med].mean(), v[v <= med].mean()
        rows.append({"plus": p, "minus": n_, "mid": (p + n_) / 2, "sep": p - n_, "asym": (p - med) - (med - n_),
                     "iso": iso[sel].mean(), "pupil": pup[sel].mean(), "speed": speed[sel].mean()})
    return pd.DataFrame(rows)
WM = {name: win_measures(Zg @ u) for name, u in AXES.items()}

def shift_spearman(a, b, rng, n=500, min_shift=10):
    ok = np.isfinite(a) & np.isfinite(b)
    a, b = a[ok], b[ok]
    if a.size < 20 or np.std(b) == 0:
        return np.nan, np.nan
    r = spearmanr(a, b).correlation
    ks = rng.integers(min_shift, a.size - min_shift, n)
    null = np.array([spearmanr(np.roll(a, k), b).correlation for k in ks])
    return r, float((1 + (np.abs(null) >= abs(r)).sum()) / (n + 1))

COVS = {"isoflurane level (whole session)": ("iso", np.ones(len(starts), bool)),
        "isoflurane level (within maintenance)": ("iso", wst == "isoflurane"),
        "pupil (within awake)": ("pupil", wst == "awake"),
        "pupil (within isoflurane)": ("pupil", wst == "isoflurane"),
        "running speed (within awake)": ("speed", wst == "awake")}
MEAS = {"+ pole": "plus", "− pole": "minus", "midpoint": "mid", "separation": "sep", "asymmetry": "asym"}
prow = []
rng = np.random.default_rng(1)
for ax_name, W in WM.items():
    for mlab, mcol in MEAS.items():
        for clab, (ccol, cm_) in COVS.items():
            r, p = shift_spearman(W[mcol].to_numpy()[cm_], W[ccol].to_numpy()[cm_], rng)
            prow.append({"axis": ax_name, "measure": mlab, "covariate": clab, "r": r, "p": p})
poles = pd.DataFrame(prow); poles.to_csv(OUT / "1_pole_measures_vs_state.csv", index=False)
for ax_name, W in WM.items():
    W.assign(centre=centres, state=wst).to_csv(OUT / f"1_pole_measures_{ax_name.split(' (')[0].replace(' ', '_')}.csv", index=False)
poles.pivot_table(index=["axis", "measure"], columns="covariate", values="r", sort=False).round(2)

covariate                          isoflurane level (whole session)  \
axis                   measure                                        
Tracker M1 (broadband) + pole                                 -0.19   
                       − pole                                 -0.85   
                       midpoint                               -0.56   
                       separation                              0.67   
                       asymmetry                               0.19   
Tracker M2             + pole                                  0.63   
                       − pole                                  0.54   
                       midpoint                                0.60   
                       separation                              0.53   
                       asymmetry                               0.61   
Tracker M3             + pole                                  0.56   
                       − pole                                  0.43   
                       midpoint                                0.55   
                       separation                              0.65   
                       asymmetry                               0.55   
Original (awake)       + pole                                 -0.02   
                       − pole                                 -0.72   
                       midpoint                               -0.41   
                       separation                              0.63   
                       asymmetry                               0.31   

covariate                          isoflurane level (within maintenance)  \
axis                   measure                                             
Tracker M1 (broadband) + pole                                      -0.68   
                       − pole                                      -0.65   
                       midpoint                                    -0.70   
                       separation                                  -0.25   
                       asymmetry                                    0.31   
Tracker M2             + pole                                      -0.40   
                       − pole                                      -0.34   
                       midpoint                                    -0.45   
                       separation                                  -0.29   
                       asymmetry                                   -0.02   
Tracker M3             + pole                                      -0.74   
                       − pole                                      -0.76   
                       midpoint                                    -0.77   
                       separation                                  -0.33   
                       asymmetry                                    0.31   
Original (awake)       + pole                                      -0.68   
                       − pole                                      -0.71   
                       midpoint                                    -0.74   
                       separation                                  -0.29   
                       asymmetry                                    0.27   

covariate                          pupil (within awake)  \
axis                   measure                            
Tracker M1 (broadband) + pole                      0.51   
                       − pole                      0.28   
                       midpoint                    0.41   
                       separation                  0.42   
                       asymmetry                   0.20   
Tracker M2             + pole                      0.01   
                       − pole                     -0.14   
                       midpoint                   -0.07   
                       separation                  0.29   
                       asymmetry                   0.07   
Tracker M3             + pole                      0.00 

In [8]:
SPAN_COL = {"awake": "#DCE9F5", "induction": "#E6E6E6", "isoflurane": "#F6DEDC"}
def shade(ax, strip=False):
    for a, b, c in [(info.t_start, info.iso_on, "awake"), (info.iso_on, info.induction_end, "induction"), (info.induction_end, info.iso_off, "isoflurane")]:
        ax.axvspan(a / 60, b / 60, color=SPAN_COL[c], lw=0, zorder=0)
    if strip:
        for kind, blocks, col in (("electrical", info.stim_blocks, "#E8955A"), ("visual", info.visual_blocks, "#6CB8A8")):
            for _, a, b in blocks:
                ax.axvspan(a / 60, b / 60, ymin=0.97, ymax=1.0, color=col, lw=0, zorder=4)

def pole_figure(ax_name, fname):
    u = AXES[ax_name]; x = Zg @ u; W = WM[ax_name]
    bins = np.linspace(np.nanpercentile(x, 0.5), np.nanpercentile(x, 99.5), 60)
    H = np.full((bins.size - 1, len(starts)), np.nan)
    for j, a in enumerate(starts):
        sel = (t >= a) & (t < a + res.config.window)
        if sel.sum() >= 150:
            h, _ = np.histogram(x[sel], bins); H[:, j] = h / max(h.max(), 1)
    fig, axs = plt.subplots(4, 1, figsize=(13, 9), sharex=True, gridspec_kw={"height_ratios": [3, 1.3, 1.1, 1.1]})
    ax = axs[0]; shade(ax, strip=True)
    ax.imshow(np.ma.masked_where(~(H > 0.02), H), aspect="auto", origin="lower", cmap="Greys", vmin=0, vmax=1, extent=[starts[0] / 60 + 2.5, starts[-1] / 60 + 2.5, bins[0], bins[-1]],
              interpolation="nearest", zorder=1, alpha=0.9)
    cm_ = centres / 60
    ax.plot(cm_, W.plus, color="#C2504A", lw=1.6, label="+ pole", zorder=2)
    ax.plot(cm_, W.minus, color="#4C8DC9", lw=1.6, label="− pole", zorder=2)
    ax.plot(cm_, W.mid, color="k", lw=1.2, ls="--", label="midpoint (baseline)", zorder=2)
    ax.set_ylabel(f"expression along {ax_name}\n(session-wide z)"); ax.legend(fontsize=8, frameon=False, ncol=3, loc="upper right")
    ax.set_title(f"{ax_name}: distribution in each 5-min window (grey, each column scaled to its peak), its two poles and their midpoint. "
                 "Background: blue awake, grey induction, red isoflurane. Top strip: orange electrical, green visual stimulation", loc="left", fontsize=8.5)
    ax = axs[1]; shade(ax)
    ax.plot(cm_, W.sep, color="#1D232B", lw=1.3, label="separation (+ pole − − pole)")
    ax2 = ax.twinx(); ax2.plot(cm_, W.asym, color="#E0A030", lw=1.1, label="asymmetry (right axis)"); ax2.axhline(0, color="#E0A030", lw=0.5, ls=":")
    ax2.spines[["top"]].set_visible(False)
    ax.legend(fontsize=8, frameon=False, loc="upper left"); ax2.legend(fontsize=8, frameon=False, loc="upper right")
    ax.set_ylabel("separation")
    ax = axs[2]; shade(ax)
    ax.plot(info.iso_t / 60, info.iso_level, color="#555555", lw=1.2); ax.set_ylabel("isoflurane (%)")
    ax = axs[3]; shade(ax)
    ax.plot(t / 60, pup, color="#8E5EA2", lw=0.5); ax.set_ylabel("pupil (px)", color="#8E5EA2")
    ax3 = ax.twinx(); ax3.plot(t / 60, pd.Series(speed).rolling(30, center=True, min_periods=1).mean(), color="#E0A030", lw=0.8)
    ax3.set_ylabel("running (cm/s, 30-s mean)", color="#E0A030"); ax3.spines[["top"]].set_visible(False)
    ax.set_xlabel("minutes"); ax.set_xlim(t[0] / 60, t[-1] / 60)
    fig.tight_layout(); save(fig, fname); plt.show()

pole_figure("Tracker M1 (broadband)", "Q1c_poles_M1")
pole_figure("Tracker M2", "Q1d_poles_M2")
pole_figure("Original (awake)", "Q1e_poles_original")

In [9]:
fig, axes = plt.subplots(1, len(AXES), figsize=(16, 4.2), sharey=True)
covs = list(COVS)
for ax, ax_name in zip(axes, AXES):
    sub = poles[poles.axis == ax_name].pivot_table(index="measure", columns="covariate", values="r", sort=False).reindex(index=list(MEAS), columns=covs)
    pp = poles[poles.axis == ax_name].pivot_table(index="measure", columns="covariate", values="p", sort=False).reindex(index=list(MEAS), columns=covs)
    ax.imshow(sub.to_numpy(), cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    for i in range(sub.shape[0]):
        for j in range(sub.shape[1]):
            r, p = sub.iat[i, j], pp.iat[i, j]
            if np.isfinite(r):
                ax.text(j, i, f"{r:.2f}" + ("*" if p <= 0.05 else ""), ha="center", va="center", fontsize=8,
                        color="white" if abs(r) > 0.6 else "k", fontweight="bold" if p <= 0.05 else "normal")
    ax.set_xticks(range(len(covs)), [c.replace(" (", "\n(") for c in covs], rotation=90, fontsize=8)
    ax.set_yticks(range(len(MEAS)), list(MEAS)); ax.set_title(ax_name, fontsize=10)
fig.suptitle("Spearman r between each pole measure and dose or behaviour, across 5-min windows (* p ≤ 0.05, circular-shift null)",
             x=0.01, ha="left", fontsize=10)
fig.tight_layout(); save(fig, "Q1f_pole_measures_vs_state"); plt.show()

## 1.4 Motif → state map: which tracker motifs are present in which state?

Presence = the tracker detected a window motif within |cos| ≥ 0.8 of the library motif in that window. The
static pipelines answer this with one yes/no per state window.

In [10]:
ws = res.window_stats.copy()
def wstate(c):
    s = sd.state_of_time(np.array([c]), info)[0]
    for _, a, b in info.stim_blocks:
        if a <= c < b: return s + " + electrical"
    for _, a, b in info.visual_blocks:
        if a <= c < b: return s + " + visual"
    return s
ws["state"] = ws.centre.map(wstate)
order = ["awake", "awake + electrical", "awake + visual", "induction", "isoflurane", "isoflurane + electrical", "isoflurane + visual"]
order = [o for o in order if o in ws.state.unique()]
pres = ws.pivot_table(index="motif", columns="state", values="present", aggfunc="mean").reindex(columns=order)
nwin = ws[ws.motif == 0].state.value_counts().reindex(order)
pres.index = [f"M{i + 1}" + (" (broadband)" if res.library[i].broadband else "") for i in pres.index]
pres.to_csv(OUT / "1_presence_by_state.csv")
fig, ax = plt.subplots(figsize=(10, 4.6))
ax.imshow(pres.to_numpy(), cmap="Greens", vmin=0, vmax=1, aspect="auto")
for i in range(pres.shape[0]):
    for j in range(pres.shape[1]):
        v = pres.iat[i, j]
        ax.text(j, i, f"{v:.0%}", ha="center", va="center", fontsize=8, color="white" if v > 0.6 else "k")
ax.set_xticks(range(len(order)), [o.replace(" + ", "\n+ ") + f"\n({nwin[o]} windows)" for o in order], fontsize=8)
ax.set_yticks(range(pres.shape[0]), pres.index)
ax.set_title("Share of 5-min windows in which each tracker motif is detected, by state", loc="left", fontsize=10)
save(fig, "Q1g_presence_by_state"); plt.show()
static_tbl = pd.DataFrame([{"pipeline": v, "state window": w, "motifs found": int(runs[w, v].shape[1])}
                           for w in ("awake", "isoflurane", "full") for v in ("strict", "relaxed")])
static_tbl.to_csv(OUT / "1_static_motifs_by_state.csv", index=False)
static_tbl

,pipeline,state window,motifs found
0,strict,awake,1
1,relaxed,awake,1
2,strict,isoflurane,0
3,relaxed,isoflurane,1
4,strict,full,0
5,relaxed,full,1


# Part 2: neural grounding

PL5 units, smoothed 1-s rates. Opponency tests remove the population mean from every neuron first (shared gain
changes would otherwise look like one pole driving everything).

In [11]:
rate, unit_ids = sv.unit_rates(NWB, "PL5", t)
masks = {st: m & spont & ok_tr & np.isfinite(sig.resid).all(1) for st, m in sv.state_masks(t, info).items()}
print(rate.shape[1], "units;", {k: int(v.sum()) for k, v in masks.items()})

63 units; {'awake': 997, 'isoflurane': 1438}


## 2.1 Like for like: the same direction in the static frame vs the tracker frame

For each direction (static motifs, tracker M1–M3, and the pure broadband direction as a reference), expression is
computed two ways on the same seconds:
- **static frame**: band envelopes z-scored within the state, projected on the direction (what the static
  pipelines use);
- **tracker frame**: the tracker's residual (envelopes minus the 5-min sliding baseline, divided by the sliding
  spread), projected on the direction.

Score: held-out opponency. Groups A (up in the + pole) and B (up in the − pole) are chosen on training blocks;
on held-out blocks we correlate expression with A − B. Null: expression circularly shifted against the neurons.

**Broadband control**: broadband power (mean of the 10 bands, same frame) is regressed out of the expression and
out of every neuron before the test. The broadband direction itself is then empty by construction.

In [12]:
DIRS = {**STATIC, **{f"Tracker M{i + 1}": U[i] for i in range(3)}, "Broadband (reference)": BROAD}
def frames(m):
    Zs = sig.Z[m]; Zs = (Zs - Zs.mean(0)) / Zs.std(0)
    return {"static frame": Zs, "tracker frame": sig.resid[m]}
def resid_on(y, b):
    b = b - b.mean()
    yc = y - y.mean(0)
    beta = (b @ yc) / (b @ b)
    return yc - np.outer(b, beta) if y.ndim == 2 else yc - b * beta

orow = []
rng = np.random.default_rng(2)
for st, m in masks.items():
    Rg = sv.remove_global(rate[m])
    for fr, Zf in frames(m).items():
        bb = Zf @ BROAD
        for dname, u in DIRS.items():
            x = Zf @ u
            ho = sv.heldout_opponency(x, Rg, rng, n_shift=200)
            orow.append({"state": st, "frame": fr, "direction": dname, "control": "none", **ho})
            if dname != "Broadband (reference)":
                hc = sv.heldout_opponency(resid_on(x, bb), resid_on(Rg, bb), rng, n_shift=200)
                orow.append({"state": st, "frame": fr, "direction": dname, "control": "broadband removed", **hc})
opp = pd.DataFrame(orow); opp.to_csv(OUT / "2_opponency_like_for_like.csv", index=False)
opp.pivot_table(index=["state", "direction"], columns=["control", "frame"], values="r_AminusB", sort=False).round(2)

control                                    none broadband removed  \
frame                              static frame      static frame   
state      direction                                                
awake      Original (awake)                0.32              0.21   
           Relaxed (awake)                 0.35              0.20   
           Relaxed (whole session)         0.25              0.18   
           Relaxed (isoflurane)           -0.01             -0.06   
           Tracker M1                      0.20              0.25   
           Tracker M2                      0.14              0.15   
           Tracker M3                      0.21              0.07   
           Broadband (reference)           0.19               NaN   
isoflurane Original (awake)                0.18              0.17   
           Relaxed (awake)                 0.18              0.20   
           Relaxed (whole session)         0.43              0.42   
           Relaxed (isoflurane)            0.34              0.41   
           Tracker M1                      0.09              0.03   
           Tracker M2                      0.49              0.47   
           Tracker M3                      0.42              0.23   
           Broadband (reference)           0.06               NaN   

control                                     none broadband removed  
frame                              tracker frame     tracker frame  
state      direction                                                
awake      Original (awake)                 0.31              0.15  
           Relaxed (awake)                  0.32              0.14  
           Relaxed (whole session)          0.21              0.17  
           Relaxed (isoflurane)             0.03              0.02  
           Tracker M1                       0.19              0.17  
           Tracker M2                       0.02              0.06  
           Tracker M3                       0.05             -0.08  
           Broadband (reference)            0.21               NaN  
isoflurane Original (awake)                 0.23              0.06  
           Relaxed (awake)                  0.24              0.06  
           Relaxed (whole session)          0.40              0.40  
           Relaxed (isoflurane)             0.25              0.31  
           Tracker M1                       0.21             -0.17  
           Tracker M2                       0.45              0.33  
           Tracker M3                       0.35              0.13  
           Broadband (reference)            0.20               NaN

In [13]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True, sharey=True)
dn = list(DIRS)
for r_, st in enumerate(masks):
    for c_, ctrl in enumerate(["none", "broadband removed"]):
        ax = axes[r_, c_]
        for k, (fr, col, off) in enumerate([("static frame", "#2F7F8F", -0.15), ("tracker frame", "#C2504A", 0.15)]):
            sub = opp[(opp.state == st) & (opp.frame == fr) & (opp.control == ctrl)].set_index("direction")
            for i, d in enumerate(dn):
                if d not in sub.index: continue
                rr = sub.loc[d]
                sig_ = rr.p_AminusB <= 0.05
                ax.scatter(i + off, rr.r_AminusB, s=55, color=col if sig_ else "white", edgecolor=col, lw=1.5, zorder=3,
                           label=fr if i == 0 else None)
                ax.plot([i + off - 0.12, i + off + 0.12], [rr.null95_AminusB] * 2, color="#777777", lw=1)
        ax.axhline(0, color="0.7", lw=0.6)
        ax.set_title(f"{st} · {'no control' if ctrl == 'none' else 'broadband power removed'}", loc="left", fontsize=10)
        ax.set_xticks(range(len(dn)), [d.replace(" (", "\n(") for d in dn], fontsize=8, rotation=0)
        if c_ == 0: ax.set_ylabel("held-out r(expression, A − B)")
axes[0, 0].legend(fontsize=8, frameon=False, loc="upper right")
fig.suptitle("Opponent neuron groups, same direction in two frames (filled: p ≤ 0.05; grey tick: 95th percentile of the shift null)",
             x=0.01, ha="left", fontsize=10)
fig.tight_layout(); save(fig, "Q2a_opponency_like_for_like"); plt.show()

## 2.2 Spiking with broadband power controlled, and what the baseline adds

Population rate and getClust ensembles: held-out R² of broadband power alone, then the gain each pipeline adds on
top of it (R²(pipeline + broadband) − R²(broadband)), with a null from time-shifted targets (50 shifts).

Broadband power is taken in the same frame as each method: within-state z for the static pipelines and the 10
bands, the tracker's residual for tracker expression. Mixing frames would let the model rebuild the slow baseline
(session-wide broadband minus residual broadband), which reintroduces the slow leak.

Only per-second predictors are used. Yesterday's "Tracker: both" (expression + baseline) was scored with
interleaved folds, under which slow 5-min features partly fit even time-shifted targets (its null was 0.17
under isoflurane and 0.34 awake). The baseline's link to neurons is tested separately in 2.4 with slow
correlations.

In [14]:
grow = []
for st, m in masks.items():
    clusters, stab, X = sv.ensembles(rate, m)
    targets = sv.ensemble_targets(X, clusters, rate[m])
    Zs = sig.Z[m]; Zs = (Zs - Zs.mean(0)) / Zs.std(0)
    bb_static, bb_tracker = (Zs @ BROAD)[:, None], (sig.resid[m] @ BROAD)[:, None]
    P = {"Original": Zs @ STATIC["Original (awake)"][:, None], "Relaxed": Zs @ Srel,
         "Tracker expression": E[m], "10 bands": Zs}
    for tn, y in targets.items():
        yz = sv.zscore(y)
        for lab, bb in (("Broadband power alone (static frame)", bb_static), ("Broadband power alone (tracker frame)", bb_tracker)):
            grow.append({"state": st, "target": tn, "features": lab, "r2": sv.cv_r2(bb, yz), "gain": np.nan})
        for pn, F in P.items():
            bbF = bb_tracker if pn.startswith("Tracker") else bb_static
            r_bb = sv.cv_r2(bbF, yz)
            r_p = sv.cv_r2(F, yz)
            r_pb = sv.cv_r2(np.column_stack([F, bbF]), yz)
            null = [sv.cv_r2(np.column_stack([F, bbF]), np.roll(yz, k)) - sv.cv_r2(bbF, np.roll(yz, k))
                    for k in rng.integers(60, yz.size - 60, 50)]
            grow.append({"state": st, "target": tn, "features": pn, "r2": r_p, "r2_with_bb": r_pb, "gain": r_pb - r_bb,
                         "gain_null95": float(np.percentile(null, 95))})
gain = pd.DataFrame(grow); gain.to_csv(OUT / "2_spiking_gain_over_broadband.csv", index=False)
gain.pivot_table(index=["state", "target"], columns="features", values=["r2", "gain"], sort=False).round(2)

r2  \
features                         Broadband power alone (static frame)   
state      target                                                       
awake      population rate                                      -0.07   
           ensemble 1 (32 units)                                -0.08   
           ensemble 2 (6 units)                                 -0.06   
isoflurane population rate                                       0.54   
           ensemble 1 (27 units)                                -0.01   
           ensemble 2 (25 units)                                 0.08   
           ensemble 3 (6 units)                                  0.09   

                                                                        \
features                         Broadband power alone (tracker frame)   
state      target                                                        
awake      population rate                                       -0.10   
           ensemble 1 (32 units)                                 -0.12   
           ensemble 2 (6 units)                                  -0.06   
isoflurane population rate                                        0.31   
           ensemble 1 (27 units)                                 -0.01   
           ensemble 2 (25 units)                                  0.07   
           ensemble 3 (6 units)                                   0.06   

                                                                               \
features                         Original Relaxed Tracker expression 10 bands   
state      target                                                               
awake      population rate           0.05   -0.05              -0.23    -0.13   
           ensemble 1 (32 units)     0.15    0.06              -0.29    -0.05   
           ensemble 2 (6 units)     -0.05   -0.12              -0.06    -0.08   
isoflurane population rate           0.40    0.61               0.35     0.68   
           ensemble 1 (27 units)    -0.02   -0.04              -0.08     0.07   
           ensemble 2 (25 units)     0.08    0.10               0.08     0.07   
           ensemble 3 (6 units)      0.08    0.13               0.07     0.12   

                                     gain                                      
features                         Original Relaxed Tracker expression 10 bands  
state      target                                                              
awake      population rate           0.04   -0.04              -0.13    -0.06  
           ensemble 1 (32 units)     0.14    0.07              -0.16     0.03  
           ensemble 2 (6 units)     -0.00   -0.06               0.01    -0.02  
isoflurane population rate           0.05    0.14               0.03     0.14  
           ensemble 1 (27 units)    -0.00    0.01              -0.06     0.08  
           ensemble 2 (25 units)    -0.01    0.01               0.01    -0.00  
           ensemble 3 (6 units)     -0.00    0.05               0.01     0.03

In [15]:
tgts = gain[["state", "target"]].drop_duplicates().values.tolist()
pn = ["Original", "Relaxed", "Tracker expression", "10 bands"]
PC2 = {"Original": "#9AA5B1", "Relaxed": "#2F7F8F", "Tracker expression": "#E0A030", "10 bands": "#B7A4C9"}
fig, axes = plt.subplots(1, 2, figsize=(15, 4.6), gridspec_kw={"width_ratios": [1, 2.2]})
ax = axes[0]
for k, (lab, col) in enumerate((("Broadband power alone (static frame)", "#BBBBBB"), ("Broadband power alone (tracker frame)", "#E0A030"))):
    bbv = [gain[(gain.state == s) & (gain.target == tg_) & (gain.features == lab)].r2.iloc[0] for s, tg_ in tgts]
    ax.barh(np.arange(len(tgts)) + (k - 0.5) * 0.4, bbv, 0.38, color=col, label=lab.replace("Broadband power alone ", ""))
ax.invert_yaxis(); ax.legend(fontsize=8, frameon=False, loc="lower right"); ax.axvline(0, color="0.6", lw=0.6)
ax.set_yticks(range(len(tgts)), [f"{s}: {tg_}" for s, tg_ in tgts], fontsize=8); ax.set_xlabel("held-out R², broadband power alone")
ax.set_title("What broadband power already explains", loc="left", fontsize=10)
ax = axes[1]
w = 0.8 / len(pn)
for j, p_ in enumerate(pn):
    for i, (s, tg_) in enumerate(tgts):
        r = gain[(gain.state == s) & (gain.target == tg_) & (gain.features == p_)].iloc[0]
        x = i + (j - (len(pn) - 1) / 2) * w
        ax.bar(x, max(r.gain, -0.1), w * 0.92, color=PC2[p_], label=p_ if i == 0 else None,
               edgecolor="k" if r.gain > r.gain_null95 else "none", lw=0.8)
        ax.plot([x - w / 2, x + w / 2], [r.gain_null95] * 2, color="#444444", lw=1)
ax.axhline(0, color="0.6", lw=0.6)
ax.set_xticks(range(len(tgts)), [f"{s}\n" + tg_.replace(" (", "\n(") for s, tg_ in tgts], fontsize=8)
ax.set_ylabel("R² gained over broadband power\n(clipped at −0.1)"); ax.legend(fontsize=8, frameon=False, ncol=5, loc="upper left")
ax.set_title("What each pipeline adds on top of broadband power (black edge: above the shift null's 95th percentile)", loc="left", fontsize=10)
fig.tight_layout(); save(fig, "Q2b_spiking_gain_over_broadband"); plt.show()

## 2.3 Why does the tracker lose the awake relation? Sweep the sliding-baseline window

Same directions, residuals rebuilt with baselines of 1, 2, 5, 10 and 20 minutes, and "none" (the static frame).
Scores: held-out opponency r(A − B) and the Mishler pole contrast for the population rate (+ pole seconds minus
− pole seconds, in z).

In [16]:
WINDOWS = [60, 120, 300, 600, 1200, None]
SWEEP_DIRS = {"Original (awake)": STATIC["Original (awake)"], "Tracker M1": U[0], "Tracker M2": U[1], "Tracker M3": U[2]}
srow = []
for bw in WINDOWS:
    if bw is None:
        resid = None
    else:
        b_, s_ = mt.split_baseline(sig.Z, sig.segment, bw)
        resid = (sig.Z - b_) / s_
    for st, m in masks.items():
        Rg = sv.remove_global(rate[m]); pop = rate[m].sum(1)
        if resid is None:
            Zf = sig.Z[m]; Zf = (Zf - Zf.mean(0)) / Zf.std(0)
        else:
            Zf = resid[m]
        for dname, u in SWEEP_DIRS.items():
            x = Zf @ u
            ho = sv.heldout_opponency(x, Rg, rng, n_shift=100)
            pc, pcp = sv.pole_contrast(x, pop, rng, n_shift=300)
            srow.append({"baseline window": "none (static frame)" if bw is None else f"{bw // 60} min", "state": st, "direction": dname,
                         "r(A−B)": ho["r_AminusB"], "p(A−B)": ho["p_AminusB"], "pop pole contrast (z)": pc, "p pole": pcp})
sweep = pd.DataFrame(srow); sweep.to_csv(OUT / "2_baseline_window_sweep.csv", index=False)
sweep.pivot_table(index=["state", "direction"], columns="baseline window", values=["r(A−B)", "pop pole contrast (z)"], sort=False).round(2)

r(A−B)                            \
baseline window              1 min 2 min 5 min 10 min 20 min   
state      direction                                           
awake      Original (awake)   0.26  0.29  0.31   0.33   0.32   
           Tracker M1         0.16  0.21  0.19   0.25   0.23   
           Tracker M2         0.00 -0.01  0.02  -0.04   0.02   
           Tracker M3         0.07  0.05  0.05   0.08   0.04   
isoflurane Original (awake)   0.24  0.24  0.23   0.22   0.23   
           Tracker M1         0.22  0.18  0.21   0.20   0.16   
           Tracker M2         0.40  0.42  0.45   0.44   0.44   
           Tracker M3         0.37  0.40  0.35   0.39   0.43   

                                                pop pole contrast (z)        \
baseline window             none (static frame)                 1 min 2 min   
state      direction                                                          
awake      Original (awake)                0.32                  0.07  0.01   
           Tracker M1                      0.20                  0.05  0.05   
           Tracker M2                      0.14                  0.03 -0.05   
           Tracker M3                      0.21                 -0.00  0.01   
isoflurane Original (awake)                0.18                  0.76  0.73   
           Tracker M1                      0.09                  0.80  0.81   
           Tracker M2                      0.49                  0.29  0.25   
           Tracker M3                      0.42                  0.55  0.48   

                                                                     
baseline window             5 min 10 min 20 min none (static frame)  
state      direction                                                 
awake      Original (awake) -0.02  -0.04   0.02                0.48  
           Tracker M1       -0.00   0.00   0.04                0.32  
           Tracker M2       -0.08  -0.13  -0.12                0.18  
           Tracker M3        0.03   0.02   0.03                0.28  
isoflurane Original (awake)  0.84   0.81   0.80                0.98  
           Tracker M1        0.93   0.93   0.95                1.21  
           Tracker M2        0.26   0.26   0.28                0.17  
           Tracker M3        0.52   0.49   0.49                0.52

In [17]:
labels = ["1 min", "2 min", "5 min", "10 min", "20 min", "none (static frame)"]
DC = {"Original (awake)": "#9AA5B1", "Tracker M1": "#1D232B", "Tracker M2": "#E0A030", "Tracker M3": "#2F7F8F"}
fig, axes = plt.subplots(2, 2, figsize=(13, 7), sharex=True)
for c_, st in enumerate(masks):
    for r_, (col, pcol, ylab) in enumerate([("r(A−B)", "p(A−B)", "held-out r(expression, A − B)"),
                                             ("pop pole contrast (z)", "p pole", "population rate, + pole − − pole (z)")]):
        ax = axes[r_, c_]
        for d in SWEEP_DIRS:
            sub = sweep[(sweep.state == st) & (sweep.direction == d)].set_index("baseline window").reindex(labels)
            ax.plot(range(len(labels)), sub[col], color=DC[d], lw=1.4, label=d)
            sigm = sub[pcol] <= 0.05
            ax.scatter(np.flatnonzero(sigm), sub[col][sigm], color=DC[d], s=40, zorder=3)
            ax.scatter(np.flatnonzero(~sigm), sub[col][~sigm], color="white", edgecolor=DC[d], s=40, zorder=3)
        ax.axhline(0, color="0.7", lw=0.6); ax.axvline(2, color="0.85", lw=6, zorder=0)
        ax.set_title(st, loc="left", fontsize=10) if r_ == 0 else None
        if c_ == 0: ax.set_ylabel(ylab)
        ax.set_xticks(range(len(labels)), labels, fontsize=8)
axes[0, 0].legend(fontsize=8, frameon=False)
fig.suptitle("Effect of the sliding-baseline window (shaded: the tracker's 5 min; filled: p ≤ 0.05)", x=0.01, ha="left", fontsize=10)
fig.tight_layout(); save(fig, "Q2c_baseline_window_sweep"); plt.show()

## 2.4 Does a baseline shift go with a shift in the neurons? All motifs, corrected

Two tests for every tracker motif in both states, each with Benjamini–Hochberg correction across the 2 × 9 tests:
- **(a) slow opponency**: opponent groups A and B are chosen from the fast parts; does the slow part of A − B
  follow the motif's baseline? Null: baseline circularly shifted by at least 5 min.
- **(b) reorganisation**: within-third coupling stability minus low-vs-high-baseline coupling similarity; above
  0 means different neurons go on and off at low and high baseline.

In [18]:
def bh(p):
    p = np.asarray(p, float); n = p.size; o = np.argsort(p)
    q = np.minimum.accumulate((p[o] * n / np.arange(1, n + 1))[::-1])[::-1]
    out = np.empty(n); out[o] = np.minimum(q, 1); return out
mrow = []
for st, m in masks.items():
    Rg = sv.remove_global(rate[m]); tt = t[m]
    for i in range(K):
        r_ = sv.membership_shift(E[m][:, i], Bm[m][:, i], Rg, tt, rng, n_shift=200)
        so = sv.slow_opponency(E[m][:, i], Bm[m][:, i], Rg, tt, rng, n_shift=500)
        mrow.append({"state": st, "motif": f"M{i + 1}" + (" (broadband)" if res.library[i].broadband else ""),
                     "baseline range": float(np.ptp(Bm[m][:, i])), **r_,
                     "slow r": so["r"], "slow p": so["p"], "slow null95": so.get("null95", np.nan), "n_A": so["n_A"], "n_B": so["n_B"]})
reorg = pd.DataFrame(mrow)
reorg["q (BH)"] = bh(reorg.p)
reorg["slow q (BH)"] = bh(reorg["slow p"].fillna(1))
reorg.to_csv(OUT / "2_baseline_vs_neurons_all_motifs.csv", index=False)
reorg.round(3)

,state,motif,baseline range,within,between,reorganisation,p,null95,slow r,slow p,slow null95,n_A,n_B,q (BH),slow q (BH)
0,awake,M1 (broadband),0.853,0.333,0.254,0.079,0.333,0.421,0.293,0.082,0.306,8,12,0.746,0.210
1,awake,M2,0.372,0.175,0.086,0.088,0.274,0.177,-0.262,0.251,0.376,2,6,0.704,0.503
2,awake,M3,0.697,0.133,0.182,-0.049,0.542,0.184,0.458,0.002,0.398,7,4,0.751,0.007
3,awake,M4,0.562,0.220,0.162,0.058,0.527,0.301,0.487,0.002,0.355,10,4,0.751,0.007
4,awake,M5,0.354,0.202,-0.077,0.279,0.204,0.475,0.015,0.934,0.138,2,5,0.687,0.934
5,awake,M6,0.762,0.358,0.291,0.067,0.642,0.219,0.561,0.002,0.362,8,4,0.825,0.007
6,awake,M7,0.388,-0.001,0.128,-0.129,0.965,0.406,0.259,0.639,0.466,8,4,0.965,0.884
7,awake,M8,1.045,0.520,0.481,0.039,0.373,0.116,0.299,0.034,0.281,12,13,0.746,0.102
8,awake,M9,0.631,0.016,-0.050,0.066,0.229,0.256,0.446,0.192,0.481,4,5,0.687,0.431
9,isoflurane,M1 (broadband),10.271,0.543,0.342,0.201,0.915,0.559,-0.039,0.697,0.359,8,7,0.965,0.896


In [21]:
fig, axes = plt.subplots(2, 2, figsize=(14, 7.5))
for c_, st in enumerate(masks):
    sub = reorg[reorg.state == st].reset_index(drop=True)
    for r_, (val, p, q, nul, ylab) in enumerate([("slow r", "slow p", "slow q (BH)", "slow null95", "(a) r(baseline, slow A − B)"),
                                                  ("reorganisation", "p", "q (BH)", "null95", "(b) reorganisation index")]):
        ax = axes[r_, c_]
        cols = ["#C2504A" if qq <= 0.05 else ("#E8B4AF" if pp <= 0.05 else "#BBBBBB") for qq, pp in zip(sub[q], sub[p].fillna(1))]
        ax.bar(range(len(sub)), sub[val].fillna(0), color=cols)
        for i, r in sub.iterrows():
            ax.plot([i - 0.4, i + 0.4], [r[nul]] * 2, color="#444444", lw=1)
            if r_ == 0: ax.plot([i - 0.4, i + 0.4], [-r[nul]] * 2, color="#444444", lw=1)
        ax.axhline(0, color="0.7", lw=0.6)
        ax.set_xticks(range(len(sub)), sub.motif.str.replace(" (broadband)", "\n(broadband)"), fontsize=8)
        if r_ == 0: ax.set_title(st, loc="left", fontsize=10)
        if c_ == 0: ax.set_ylabel(ylab)
fig.suptitle("Baseline shifts vs neurons, every tracker motif (dark red: q ≤ 0.05 after correction; pink: p ≤ 0.05 uncorrected only; "
             "ticks: null 95th percentile)", x=0.01, ha="left", fontsize=10)
fig.tight_layout(); save(fig, "Q2d_baseline_vs_neurons_all_motifs") 
plt.show()

## Conclusions

**Q1, states.**
- The tracker follows state better than the static motifs because of its sliding baseline, not because of its motif
  directions. A 5-min baseline on the raw bands, with no motifs, does about as well.
- The new finding is in the poles. Within isoflurane maintenance, a higher dose pushes the + pole, the − pole and
  their midpoint down together (r ≈ −0.7), while their separation stays the same. So dose changes move the
  baseline, not the motif.
- Between awake and isoflurane, the separation grows. That is the known increase in variance.

**Q2, neurons.**
- The tracker's frame does not track opponent neurons better. In like-for-like tests, effect sizes match the
  static frame.
- What predicts opponency is the motif direction:
  - awake: the original/relaxed awake motif;
  - isoflurane: tracker M2, which is nearly the same direction as the relaxed isoflurane motif (cos 0.93).
- Beyond broadband power, per-second tracker expression adds little to population spiking.
- The baseline does carry neural information: the slow opponent-group drift follows the baseline for 5 of 18
  motif × state pairs after correction. Which neurons are involved does not reorganise.

**Method note.** Multi-feature regressions on slow 5-min features are not testable within one session: they fit
time-shifted targets too. Yesterday's "Tracker: both" spiking number (R² 0.67) is partly inflated by this; its
null was 0.17. The tracker's per-second expression result (R² 0.35 under isoflurane, null −0.02) stands.

Figure captions for slides: `data/out/2026-11-10_tracker_vs_static/figures/CAPTIONS.md`.